# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d01_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # sediment_load train
    shutil.copy(glob.glob("/kaggle/input/**/d01_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # sediment_load test
    print("Kaggle: sediment_load data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: sediment_load data copied to ./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 mapie==0.6.0 puncc==0.8.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from pytorch_tabnet import TabNetRegressor

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795379283.984530      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795379304.327344      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
# Go to find & replace button and replace (sediment_load_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [5]:
feature_names = ['Qt', 'Qt-1', 'St-1']
TARGET_LABEL = 'Sediment load (St)'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [6]:
train_data_path = "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (1095, 4)
First 5 rows of training data:
       Qt   Qt-1    St-1      St
0  12.40  11.80  5930.0  5630.0
1   7.93  12.40  5630.0  1960.0
2   5.66   7.93  1960.0   491.0
3   5.04   5.66   491.0   692.0
4   4.84   5.04   692.0   452.0

Shape of test data: (365, 4)
First 5 rows of test data:
      Qt  Qt-1  St-1    St
0  5.27  5.49  45.0  44.0
1  5.13  5.27  44.0  29.0
2  4.84  5.13  29.0  23.0
3  5.04  4.84  23.0  26.0
4  5.21  5.04  26.0  30.0


In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (1095, 3)
Shape of y_train: (1095,)
Shape of X_test: (365, 3)
Shape of y_test: (365,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[-0.13088871 -0.15525695  1.01508124]
 [-0.31063021 -0.131129    0.95334642]
 [-0.40190834 -0.31088224  0.19812377]
 [-0.42683893 -0.40216633 -0.10417107]
 [-0.43488105 -0.42709855 -0.06280874]]

First five rows of normalized X_test:
[[-0.41759048 -0.40900259 -0.19595018]
 [-0.42321997 -0.4178495  -0.19615596]
 [-0.43488105 -0.42347936 -0.1992427 ]
 [-0.42683893 -0.4351412  -0.2004774 ]
 [-0.42000312 -0.42709855 -0.19986005]]


# **Functions:**

In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [12]:
# FIX Q2: quantile crossing. Each quantile level is a separate model, so for some rows the
# predictions can come out in the wrong order (e.g. q0.10 < q0.05). Before any metric we sort
# each row's quantile predictions (monotone rearrangement, Chernozhukov et al. 2010) and record
# how many rows crossed, for the Excel report.
quantile_crossing_report = {}

def rearrange_quantiles(predictions_df, tag=None):
    q_cols = sorted(c for c in predictions_df.columns if c != 'Actual')
    values = predictions_df[q_cols].to_numpy(dtype=float)
    crossed = (np.diff(values, axis=1) < 0).any(axis=1)
    if tag is not None:
        quantile_crossing_report[tag] = {'rows': int(len(values)), 'rows_crossing': int(crossed.sum()),
                                         'pct_crossing': round(100.0 * crossed.mean(), 2)}
    out = predictions_df.copy()
    out[q_cols] = np.sort(values, axis=1)
    return out


In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def quantile_regression_print(predictions_df, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Extract actual values
    actual_values = predictions_df['Actual'].values

    # Debugging: Print the first few actual values
    print("First few actual values:", actual_values[:5])

    # Calculate coverage for the specific interval (0.05 to 0.95)
    lower_quantile = 0.05
    median_quantile = 0.5
    upper_quantile = 0.95

    lower_preds = predictions_df[lower_quantile].values
    median_preds = predictions_df[median_quantile].values
    upper_preds = predictions_df[upper_quantile].values

    in_interval = ((actual_values >= lower_preds) & (actual_values <= upper_preds))
    coverage = in_interval.mean()
    print(f"Coverage of 90% prediction interval: {coverage * 100:.2f}%")

    # Create a new Excel workbook and add a worksheet
    wb = Workbook()
    ws = wb.active
    ws.title = "quantile_regression_print"

    # Plot actual vs. all predicted quantiles
    num_points = len(predictions_df)
    indices = np.arange(num_points)

    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values[:num_points], label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')

    # Loop over all columns assumed to be quantile predictions
    for column in predictions_df.columns:
        if column != 'Actual':  # Skip the actual values column
            plt.plot(indices, predictions_df[column][:num_points],
                     label=f'Predicted Quantile {column}', linestyle='--')

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title(f'Actual vs. Predicted Quantile {TARGET_LABEL}')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A1')

    # Plot actual values and specific prediction intervals (0.05, 0.5, 0.95)
    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values, label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')  # Connect points with a line
    plt.plot(indices, median_preds, label=f'Median Prediction ({median_quantile})', marker='x', linestyle='-', color='blue')
    plt.fill_between(
        indices,
        lower_preds,
        upper_preds,
        color='blue',
        alpha=0.5,
        label=f'Prediction Interval ({lower_quantile}-{upper_quantile})'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(f' {TARGET_LABEL}')
    plt.title(f'Actual {TARGET_LABEL} with Prediction Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1, 1))

    # Save the second plot to another BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the second image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A30')  # Adjust the cell position as needed

    # Save the workbook
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluation_features_print(predictions_df, X_test, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Ensure X_test is a DataFrame
    if isinstance(X_test, np.ndarray):
        X_test = pd.DataFrame(X_test, columns=feature_names)
    X_test.columns = feature_names

    # Get quantile columns from predictions_df except 'Actual'
    quantile_columns = [col for col in predictions_df.columns if col != 'Actual']

    # Check if the Excel file already exists
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="New Evaluation Features")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "evaluation_features_print"

    image_row = 1  # Where to start images

    for feature_to_plot in feature_names:
        feature_values = X_test[feature_to_plot]
        plot_df = predictions_df.copy()
        plot_df[feature_to_plot] = feature_values.values

        # Sort by current feature to plot lines well
        plot_df = plot_df.sort_values(by=feature_to_plot)

        # Begin Matplotlib plot
        plt.figure(figsize=(6, 4))
        # Plot each quantile as a line
        for quantile in quantile_columns:
            plt.plot(plot_df[feature_to_plot], plot_df[quantile], label=f"{quantile}")

        # Plot actual values as scatter
        plt.scatter(plot_df[feature_to_plot], plot_df['Actual'], color='black', s=12, label=f"Actual {TARGET_LABEL}")

        plt.title(f"Predicted Quantiles vs. {feature_to_plot}")
        plt.xlabel(feature_to_plot)
        plt.ylabel(f"{TARGET_LABEL}")
        plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
        plt.tight_layout()

        # Save figure to a BytesIO stream
        img_bytes = BytesIO()
        img_bytes
        plt.savefig(img_bytes, format="png")
        plt.close()
        img_bytes.seek(0)

        img = Image(img_bytes)
        img.anchor = f"A{image_row}"
        ws.add_image(img)

        image_row += 20  # Adjust as you like

    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluate_uncertainity_matrix(predictions_df, quantiles, excel_file_path, model_name='Model'):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2

   # Extract the actual target values
    y_true = predictions_df['Actual']

    # Open existing Excel file
    wb = load_workbook(_ensure_excel_file(excel_file_path))

    # Create unique sheet names
    def get_unique_sheet_name(wb, base_name):
        sheet_name = base_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base_name}_{i}"
            i += 1
        return sheet_name

    # Create separate sheets for plots and values
    plots_sheet_title = get_unique_sheet_name(wb, 'Plots')
    values_sheet_title = get_unique_sheet_name(wb, 'Values')

    ws_plots = wb.create_sheet(title=plots_sheet_title)
    ws_values = wb.create_sheet(title=values_sheet_title)
    # FIX Q2: how often the raw quantile predictions crossed (before sorting)
    _cr = quantile_crossing_report.get(excel_file_path, {})
    ws_values.append(['Rows with quantile crossing (before sorting)', _cr.get('rows_crossing'), 'of', _cr.get('rows'), 'rows', f"{_cr.get('pct_crossing')}%"])
    print(f"Quantile crossing before sorting: {_cr.get('rows_crossing')} of {_cr.get('rows')} rows ({_cr.get('pct_crossing')}%)")

    # Positions to place images
    image_positions = ['A1', 'A25', 'A49', 'A73']  # Adjust as needed

    ### 1. Validity / Coverage ###

    # Compute empirical coverage for each quantile
    empirical_quantiles = []
    for q in quantiles:
        coverage = (predictions_df[q] >= y_true).mean()
        empirical_quantiles.append(coverage)

    # Plot theoretical quantiles vs empirical quantiles
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=quantiles, y=quantiles, color="magenta", linestyle='--', linewidth=2, label="Ideal")
    sns.lineplot(x=quantiles, y=empirical_quantiles, color="blue", linewidth=2, label="Empirical")
    sns.scatterplot(x=quantiles, y=empirical_quantiles, color="blue", s=50)
    plt.xlabel("Nominal Quantile Levels")
    plt.ylabel("Empirical Quantile Levels")
    plt.title(f"Validity / Coverage - {model_name}")
    plt.legend()

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add the image to the plots worksheet
    ws_plots.add_image(img, image_positions[0])

    ### 2. Sharpness / Interval Length ###

    # Define coverage levels to evaluate
    coverage_levels = [0.6, 0.8, 0.9]
    average_interval_lengths = []

    # Compute average interval lengths for each coverage level
    for c in coverage_levels:
        lower_q = (1 - c) / 2
        upper_q = 1 - lower_q
        # Find the nearest quantiles in our quantiles list
        lower_quantile = min(quantiles, key=lambda x: abs(x - lower_q))
        upper_quantile = min(quantiles, key=lambda x: abs(x - upper_q))

        # Compute interval length for each observation
        interval_length = predictions_df[upper_quantile] - predictions_df[lower_quantile]
        avg_interval_length = interval_length.mean()
        average_interval_lengths.append(avg_interval_length)
        # Write interval length information to the values worksheet
        ws_values.append([f'Coverage level: {c*100:.0f}%', f'Interval: [{lower_quantile}, {upper_quantile}]', f'Average Interval Length: {avg_interval_length:.4f}'])

    # Plot coverage levels vs average interval lengths
    plt.figure(figsize=(10, 6))
    plt.plot([c*100 for c in coverage_levels], average_interval_lengths, marker='o')
    plt.xlabel('Coverage Level (%)')
    plt.ylabel('Average Interval Length')
    plt.title(f'Sharpness / Interval Length - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[1])

    ### 3. Negative Log-Likelihood (NLL) ###

    # Use the 5th and 95th percentiles to estimate standard deviation
    if 0.05 in quantiles and 0.95 in quantiles:
        z_lower = norm.ppf(0.05)  # z-score for 5% quantile (~ -1.6449)
        z_upper = norm.ppf(0.95)  # z-score for 95% quantile (~ +1.6449)

        # Extract quantile predictions
        q_lower = predictions_df[0.05]
        q_upper = predictions_df[0.95]

    elif 0.1 in quantiles and 0.9 in quantiles:
        z_lower = norm.ppf(0.1)  # z-score for 10% quantile (~ -1.2816)
        z_upper = norm.ppf(0.9)  # z-score for 90% quantile (~ +1.2816)

        # Extract quantile predictions
        q_lower = predictions_df[0.1]
        q_upper = predictions_df[0.9]
    else:
        raise ValueError("Required quantiles for NLL estimation not found in quantiles list.")

    # Use median as the mean estimate
    if 0.5 in quantiles:
        q_median = predictions_df[0.5]
    else:
        # If 0.5 quantile is not available, use the middle quantile
        q_median = predictions_df[quantiles[len(quantiles)/2]]

    # Estimate standard deviation for each observation
    std_estimates = (q_upper - q_lower) / (z_upper - z_lower)

    # Ensure standard deviations are positive and non-zero
    std_estimates = std_estimates.clip(lower=1e-6)

    # Extract mean estimates (median predictions)
    mean_estimates = q_median

    # Compute Negative Log-Likelihood for each observation
    nll = -norm.logpdf(y_true, loc=mean_estimates, scale=std_estimates)

    # Compute average NLL
    average_nll = nll.mean()
    # Write average NLL to the values worksheet
    ws_values.append(['Average Negative Log-Likelihood (NLL)', average_nll])

    # Write NLL values to the values worksheet
    ws_values.append(['NLL Values'])
    for value in nll:
        ws_values.append([value])

    # Plot ECDF of NLL values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(nll, color='blue', linewidth=2)
    plt.xlabel('Negative Log-Likelihood (NLL)')
    plt.ylabel('ECDF')
    plt.title(f'NLL ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[2])

    ### 4. Continuous Ranked Probability Score (CRPS) ###

    # Prepare ensemble of quantile predictions for each observation
    ensemble_predictions = predictions_df[quantiles].to_numpy()

    # Ensure quantile predictions are sorted for each observation
    ensemble_predictions.sort(axis=1)

    # Compute CRPS for each observation using ensemble predictions
    # FIX Q4: CRPS from a set of quantiles = 2 * integral of the pinball loss over tau.
    # Each quantile level gets a weight equal to the share of [0, 1] it represents
    # (levels are unevenly spaced). Was: ps.crps_ensemble, which treats the 13 quantiles as
    # 13 equally likely samples.
    _taus = np.array(quantiles, dtype=float)
    _edges = np.concatenate([[0.0], (_taus[1:] + _taus[:-1]) / 2.0, [1.0]])
    _w = np.diff(_edges)
    _err = np.asarray(y_true, dtype=float)[:, None] - ensemble_predictions
    _pinball = np.maximum(_taus * _err, (_taus - 1.0) * _err)
    crps_values = 2.0 * (_pinball * _w).sum(axis=1)

    # Compute average CRPS
    average_crps = crps_values.mean()
    # Write average CRPS to the values worksheet
    ws_values.append(['Average Continuous Ranked Probability Score (CRPS)', average_crps])

    # Write CRPS values to the values worksheet
    ws_values.append(['CRPS Values'])
    for value in crps_values:
        ws_values.append([value])

    # Plot ECDF of CRPS values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(crps_values, color='green', linewidth=2)
    plt.xlabel('Continuous Ranked Probability Score (CRPS)')
    plt.ylabel('ECDF')
    plt.title(f'CRPS ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[3])

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    wb.save(_ensure_parent_dir(excel_file_path))

    # Return a dictionary of results
    results = {
        'empirical_quantiles': empirical_quantiles,
        'average_interval_lengths': average_interval_lengths,
        'coverage_levels': coverage_levels,
        'average_nll': average_nll,
        'average_crps': average_crps,
        'nll_values': nll,
        'crps_values': crps_values
    }

    return results

In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def calibrate_and_plot_intervals(predictions_df, excel_file_path, alpha=0.1, max_samples=1000, random_seed=42):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2
    # Extract necessary columns
    lower = predictions_df[0.05].fillna(0)
    upper = predictions_df[0.95].fillna(0)
    pred = predictions_df[0.5].fillna(0)
    labels = predictions_df['Actual'].fillna(0)

    # Problem setup
    total_samples = labels.shape[0]
    n = min(max_samples, total_samples)  # Ensure n does not exceed the number of samples

    # Ensure there are enough samples for both calibration and validation
    if total_samples <= n:
        n = total_samples // 2

    # Split the data into calibration and validation sets
    np.random.seed(random_seed)  # Set the random seed for reproducibility
    idx = np.array([1] * n + [0] * (total_samples - n)) > 0
    np.random.shuffle(idx)
    cal_labels, val_labels = labels[idx], labels[~idx]
    cal_upper, val_upper = upper[idx], upper[~idx]
    cal_lower, val_lower = lower[idx], lower[~idx]
    cal_pred, val_pred = pred[idx], pred[~idx]

    # Calculate uncertainty intervals
    cal_U = cal_upper - cal_lower
    val_U = val_upper - val_lower

    # Avoid division by zero by replacing zero intervals with a small number
    cal_U[cal_U == 0] = np.finfo(float).eps
    val_U[val_U == 0] = np.finfo(float).eps

    # Get scores
    cal_scores = np.abs(cal_pred - cal_labels) / cal_U

    # Debugging: Check for NaN in scores
    if np.isnan(cal_scores).any():
        print("NaN values found in cal_scores. Check data integrity.")

    # Get the score quantile
    qhat = np.quantile(cal_scores, np.ceil((n + 1) * (1 - alpha)) / n, interpolation='higher')

    # Deploy (output=lower and upper adjusted quantiles)
    prediction_sets = [val_pred - val_U * qhat, val_pred + val_U * qhat]

    # Calculate empirical coverage (before and after calibration)
    prediction_sets_uncalibrated = [val_lower, val_upper]
    empirical_coverage_uncalibrated = ((val_labels >= prediction_sets_uncalibrated[0]) & (val_labels <= prediction_sets_uncalibrated[1])).mean() * 100
    print(f"The empirical coverage before calibration is: {empirical_coverage_uncalibrated:.2f}%")

    empirical_coverage = ((val_labels >= prediction_sets[0]) & (val_labels <= prediction_sets[1])).mean() * 100
    print(f"The empirical coverage after calibration is: {empirical_coverage:.2f}%")

    # Create a DataFrame with all the necessary data
    data = pd.DataFrame({
        'Index': np.arange(len(val_labels)),
        'Uncalibrated Lower': val_lower,
        'Uncalibrated Upper': val_upper,
        'Calibrated Lower': prediction_sets[0],
        'Calibrated Upper': prediction_sets[1],
        'True Label': val_labels,
        'Predicted Value': val_pred
    })

    plt.figure(figsize=(14, 8))

    # Plot the uncalibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Uncalibrated Lower'],
        data['Uncalibrated Upper'],
        color='blue',
        alpha=0.5,
        label='Uncalibrated Prediction Interval'
    )

    # Plot the calibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Calibrated Lower'],
        data['Calibrated Upper'],
        color='lightgreen',
        alpha=0.5,
        label='Calibrated Prediction Interval'
    )

    # Plot the true labels as points and connect them with a line
    plt.plot(
        data['Index'],
        data['True Label'],
        'o-',
        color='red',
        markersize=4,
        label='Actual Value'
    )

    # Plot the predicted values as points and connect them with a line
    plt.plot(
        data['Index'],
        data['Predicted Value'],
        'o-',
        color='black',
        markersize=4,
        label='Predicted Value'
    )

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title('Calibrated and Uncalibrated Prediction Intervals with True and Predicted Values')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()

    # Save the plot as an image
    image_path = 'plot.png'
    _ensure_parent_dir(image_path)
    plt.savefig(_ensure_parent_dir(image_path))
    plt.close()

    # Load the existing Excel file
    workbook = load_workbook(_ensure_excel_file(excel_file_path))

    # Create a new sheet for the plot
    new_sheet_name = "calibrate_and_plot_intervals"
    workbook.create_sheet(title=new_sheet_name)

    # Insert the image into the new sheet
    sheet = workbook[new_sheet_name]
    img = Image(image_path)
    sheet.add_image(img, 'A1')

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    workbook.save(_ensure_parent_dir(excel_file_path))

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def plot_quantile_intervals_to_excel(predictions_df, file_path, alpha=0.1):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    """
    Plots conformal prediction intervals and their coverage, and saves the plots to an Excel file.

    Parameters:
    - predictions_df: DataFrame containing prediction quantiles and actual values.
    - file_path: Path to the Excel file where plots will be saved.
    - alpha: Desired coverage level (default is 0.1).
    """
    # Extract mean prediction and uncertainty
    mean_prediction = predictions_df[0.5]
    uncertainty = predictions_df[0.95] - predictions_df[0.05]
    actual_values = predictions_df['Actual']

    # Calculate conformal scores
    conformal_scores = np.abs(mean_prediction - actual_values) / uncertainty

    # Function to find weighted quantile
    def weighted_quantile(values, quantile, sample_weight=None):
        values = np.array(values)
        if sample_weight is None:
            sample_weight = np.ones(len(values))
        sorter = np.argsort(values)
        values, sample_weight = values[sorter], sample_weight[sorter]
        weighted_quantiles = np.cumsum(sample_weight) - 0.5 * sample_weight
        weighted_quantiles /= np.sum(sample_weight)
        return np.interp(quantile, weighted_quantiles, values)

    # Calculate weighted quantile
    weights = np.ones_like(conformal_scores)  # Uniform weights for simplicity
    quantile = weighted_quantile(conformal_scores, 1 - alpha, sample_weight=weights)

    # Calculate prediction intervals
    lower_bound = mean_prediction - quantile * uncertainty
    upper_bound = mean_prediction + quantile * uncertainty

    # Naive conformal prediction
    naive_quantile = np.quantile(conformal_scores, 1 - alpha)
    naive_lower_bound = mean_prediction - naive_quantile * uncertainty
    naive_upper_bound = mean_prediction + naive_quantile * uncertainty

    # Coverage calculation (not plotted, just for potential diagnostics)
    weighted_coverage_points = (actual_values >= lower_bound) & (actual_values <= upper_bound)
    naive_coverage_points = (actual_values >= naive_lower_bound) & (actual_values <= naive_upper_bound)

    # Visualization using Matplotlib
    fig, ax = plt.subplots(figsize=(10, 5))

    x = np.arange(len(predictions_df))
    # Actual Values
    ax.plot(x, actual_values, label='Actual', color='black', linewidth=2)

    # Mean Prediction
    ax.plot(x, mean_prediction, label='Mean Prediction', color='darkorange', linewidth=2)

    # Weighted Prediction Interval (blue fill)
    ax.fill_between(x, lower_bound, upper_bound, color='blue', alpha=0.3, label='Weighted Interval')
    # Optionally, plot the bounds as lines if you want:
    # ax.plot(x, lower_bound, color='blue', linewidth=1, linestyle='--')
    # ax.plot(x, upper_bound, color='blue', linewidth=1, linestyle='--')

    # Naive Prediction Interval (red fill)
    ax.fill_between(x, naive_lower_bound, naive_upper_bound, color='red', alpha=0.3, label='Naive Interval')
    # Optionally, plot the bounds as lines:
    # ax.plot(x, naive_lower_bound, color='red', linewidth=1, linestyle='--')
    # ax.plot(x, naive_upper_bound, color='red', linewidth=1, linestyle='--')

    ax.set_title('Quantile Prediction Intervals')
    ax.set_xlabel('Sample Number')
    ax.set_ylabel(f'{TARGET_LABEL}')
    ax.grid(False)
    ax.set_facecolor('white')

    # Place the legend outside the plot
    ax.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), borderaxespad=0.0)

    plt.tight_layout(rect=[0,0,0.80,1.0])  # Make space for legend

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Add image to Excel
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="plot_quantile_intervals_to_excel")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "plot_quantile_intervals_to_excel"

    img = Image(img_data)
    img.anchor = 'A1'
    ws.add_image(img)
    ws.column_dimensions['A'].width = 30
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

# **Tuned Parameters**

In [18]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshjangra/sediment-load-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


In [19]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 6162139.943307468,
  'cv_rmse': 2482.365795628732,
  'best_params': {'n_estimators': 500,
   'criterion': 'squared_error',
   'max_depth': 10,
   'min_samples_split': 5,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 1.0,
   'max_leaf_nodes': 100,
   'min_impurity_decrease': 0.1,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.0},
  'test_mse': 276377.1029482182,
  'test_rmse': np.float64(525.7158005502765),
  'test_corr_coef': np.float64(0.9555378325000425),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 6162139.943307468,
  'cv_rmse': 2482.365795628732,
  'best_params': {'n_estimators': 500,
   'criterion': 'squared_error',
   'max_depth': 10,
   'min_samples_split': 5,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 1.0,
   'max_leaf_nodes': 100,
   'min_impurity_decrease': 0.1,
   'n_jo

# **Quantile Regression with Lightgbm**

In [20]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_LGBM_df = pd.DataFrame()


# Train a model for each quantile and make predictions using LightGBM
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = LGBMRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_LGBM_df[q] = predictions

# Add actual target values to the DataFrame
predictions_LGBM_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_LGBM_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  22.677197  26.076228  28.769874  30.611652  36.142411  38.527224   
1  21.319543  24.492430  26.620624  29.341409  33.807917  37.552234   
2  14.997481  17.480948  20.635200  21.416003  23.125917  25.603270   
3  10.819337  12.543326  14.732694  16.430295  19.741659  21.388613   
4  14.047410  15.966607  18.644071  19.643373  20.467904  22.107278   

         0.5        0.6        0.7        0.8       0.85         0.9  \
0  40.483014  45.000073  46.782026  58.731460  93.822456  145.135193   
1  39.778669  42.095917  46.155146  60.914927  93.987711  145.135193   
2  27.875473  29.027079  30.716834  42.549439  65.613321  129.733179   
3  23.362222  25.561541  29.782219  40.802588  64.157223  129.733179   
4  25.111845  26.993115  30.342944  43.444612  65.613321  129.733179   

         0.95  Actual  
0  392.710308    44.0  
1  392.710308    29.0  
2  392.710308    23.0  
3  392.710308    26.0  
4  392.710308    30.

In [21]:
quantile_regression_print(predictions_LGBM_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

First few actual values: [44. 29. 23. 26. 30.]
Coverage of 90% prediction interval: 83.84%


In [22]:
evaluation_features_print(predictions_LGBM_df,x_test,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

In [23]:
evaluate_uncertainity_matrix(predictions_LGBM_df, quantiles, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/LightGBM.xlsx", model_name='LightGBM')

Quantile crossing before sorting: 84 of 365 rows (23.01%)


{'empirical_quantiles': [np.float64(0.15342465753424658),
  np.float64(0.2054794520547945),
  np.float64(0.25205479452054796),
  np.float64(0.2821917808219178),
  np.float64(0.4136986301369863),
  np.float64(0.5424657534246575),
  np.float64(0.6438356164383562),
  np.float64(0.7534246575342466),
  np.float64(0.8383561643835616),
  np.float64(0.9095890410958904),
  np.float64(0.9424657534246575),
  np.float64(0.9698630136986301),
  np.float64(0.9917808219178083)],
 'average_interval_lengths': [np.float64(338.6990698792783),
  np.float64(601.915670086881),
  np.float64(1000.8208461465508)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(6.127736738912803),
 'average_crps': np.float64(88.0008323953333),
 'nll_values': array([ 5.64222126,  5.64995251,  5.6631757 ,  5.67353326,  5.66568802,
         5.6544532 ,  5.65518691,  5.66874118,  5.68054507,  5.67977996,
         5.66499371,  5.67700898,  5.67973516,  5.67484183,  5.67495044,
         5.74283227, 14.20978628, 14.154

In [24]:
calibrate_and_plot_intervals(predictions_LGBM_df, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

The empirical coverage before calibration is: 82.51%
The empirical coverage after calibration is: 91.80%


In [25]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_LGBM_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

# **Quantile Regression with XGBoost**

In [26]:
# Get the best hyperparameters for XGBRegressor
best_paramst = get_best_model_params(best_scores_autosampler, 'XGBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_XGB_df = pd.DataFrame()

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_paramst.copy()  # FIX Q1: was best_params (= LightGBM's settings from the cell above)

    # Adjust according to your XGBoost version
    params.update({
        'objective': 'reg:quantileerror',  # Use the correct objective
        'quantile_alpha': q,                         # Use 'alpha' to set the quantile level
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = XGBRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_XGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_XGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_XGB_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  23.782700  24.998545  31.598219  32.678917  36.059319  38.221050   
1  22.369057  22.436804  24.785856  29.138306  35.533028  37.400902   
2  13.750690  17.580900  19.954378  22.853409  23.791698  29.709869   
3   9.334647  12.559442  15.193020  17.297235  21.122913  21.146805   
4  12.834319  14.146750  20.152250  20.733652  22.672928  22.855480   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  36.323067  39.653988  43.838741  54.259369  60.769810  59.294758   
1  36.978996  37.909111  48.951668  55.722992  40.939476  53.144932   
2  25.366337  28.402678  30.651815  27.017174  28.751465  28.478045   
3  22.751228  25.823437  25.728275  25.376942  28.214933  24.150265   
4  26.863167  29.009638  27.378716  24.843012  29.462122  31.014217   

        0.95  Actual  
0  59.193180    44.0  
1  55.366138    29.0  
2  38.503170    23.0  
3  34.599255    26.0  
4  34.599255    30.0  


In [27]:
quantile_regression_print(predictions_XGB_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

First few actual values: [44. 29. 23. 26. 30.]
Coverage of 90% prediction interval: 81.10%


In [28]:
evaluation_features_print(predictions_XGB_df,x_test,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

In [29]:
evaluate_uncertainity_matrix(predictions_XGB_df, quantiles, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/XGBoost.xlsx", model_name='XGBoost')

Quantile crossing before sorting: 355 of 365 rows (97.26%)


{'empirical_quantiles': [np.float64(0.13424657534246576),
  np.float64(0.18356164383561643),
  np.float64(0.23013698630136986),
  np.float64(0.29041095890410956),
  np.float64(0.38904109589041097),
  np.float64(0.4767123287671233),
  np.float64(0.5698630136986301),
  np.float64(0.663013698630137),
  np.float64(0.7205479452054795),
  np.float64(0.8027397260273973),
  np.float64(0.8356164383561644),
  np.float64(0.8794520547945206),
  np.float64(0.9452054794520548)],
 'average_interval_lengths': [np.float64(285.6988132241654),
  np.float64(687.5968628223628),
  np.float64(1176.0468888766145)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(6.05575438710404),
 'average_crps': np.float64(83.75527606300786),
 'nll_values': array([  3.47080296,   3.57859071,   3.07958975,   3.04701864,
          3.11222214,   3.10520653,   3.27098981,   3.60692417,
          3.24361396,   3.83264849,   3.67388256,   3.37265979,
          2.96321236,   2.95787638,   2.96299882,   5.90563629,


In [30]:
calibrate_and_plot_intervals(predictions_XGB_df, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

The empirical coverage before calibration is: 80.33%
The empirical coverage after calibration is: 92.90%


In [31]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_XGB_df, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

# **Quantile Regression with GPBoost**

In [32]:
# Get the best hyperparameters for GPBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GPBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'alpha', 'quantile_alpha', 'boosting_type', 'metric', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = GPBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GPBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GPBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GPBoost_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  21.644311  23.335931  31.058766  34.021443  34.642583  35.615365   
1  21.347479  23.426211  26.010600  27.170733  32.882614  33.035016   
2  15.495495  19.840893  17.347363  20.947687  22.508188  24.600609   
3  10.866949  12.084353  14.980225  16.302785  19.165771  21.314950   
4  14.688346  14.905970  18.008488  20.494286  20.225892  21.314950   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  39.778362  42.544635  45.858717  46.860622  76.822061  83.376948   
1  39.822019  42.912984  42.903023  45.666998  76.408335  83.376948   
2  27.004725  28.016747  28.878131  28.740087  35.594668  63.162854   
3  24.636855  27.360562  35.395739  31.685134  34.544630  63.162854   
4  28.295531  27.180131  28.270001  28.986735  35.356906  63.162854   

         0.95  Actual  
0  161.410686    44.0  
1  161.410686    29.0  
2  161.410686    23.0  
3  161.410686    26.0  
4  161.410686    30.0  


In [33]:
quantile_regression_print(predictions_GPBoost_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

First few actual values: [44. 29. 23. 26. 30.]
Coverage of 90% prediction interval: 84.11%


In [34]:
evaluation_features_print(predictions_GPBoost_df,x_test,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

In [35]:
evaluate_uncertainity_matrix(predictions_GPBoost_df, quantiles,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/GPBoost.xlsx", model_name='GPBoost')

Quantile crossing before sorting: 250 of 365 rows (68.49%)


{'empirical_quantiles': [np.float64(0.13972602739726028),
  np.float64(0.19726027397260273),
  np.float64(0.2602739726027397),
  np.float64(0.31232876712328766),
  np.float64(0.39452054794520547),
  np.float64(0.4904109589041096),
  np.float64(0.5945205479452055),
  np.float64(0.6958904109589041),
  np.float64(0.7863013698630137),
  np.float64(0.8712328767123287),
  np.float64(0.9260273972602739),
  np.float64(0.9561643835616438),
  np.float64(0.9808219178082191)],
 'average_interval_lengths': [np.float64(246.47356351554097),
  np.float64(542.1151647803126),
  np.float64(767.8565587890488)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(5.524051206097259),
 'average_crps': np.float64(75.58976422687448),
 'nll_values': array([ 4.67304898,  4.7025375 ,  4.71524147,  4.74283726,  4.71868062,
         4.70648896,  4.70929529,  4.73628771,  4.77303586,  4.77543403,
         4.73419533,  4.74987497,  4.76745347,  4.74747245,  4.74787143,
         5.11294155, 58.03169091, 10

In [36]:
calibrate_and_plot_intervals(predictions_GPBoost_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

The empirical coverage before calibration is: 83.61%
The empirical coverage after calibration is: 92.90%


In [37]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GPBoost_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

# **Quantile Regression with NGBoost**

In [38]:
# Correct best_params assignment
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# DataFrame to store predictions
predictions_NGB_df = pd.DataFrame()

# Remove 'Score' and 'Dist' if present in best_params
for key in ['Score', 'Dist']:
    best_params.pop(key, None)

# Update params with model settings
best_params.update({
    'Dist': Normal,
    'Score': LogScore,
    'random_state': 42,
    'verbose': True
})

# Initialize and fit the NGBoost model once
model = NGBRegressor(**best_params)

# Fit the model
model.fit(X_train, y_train)

# Make predictions
pred_dist = model.pred_dist(X_test)

# Extract the mean (mu) and standard deviation (sigma) of the predicted distributions
mu = pred_dist.loc    # Mean predictions
sigma = pred_dist.scale  # Standard deviation predictions

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# For each quantile, compute predictions
for q in quantiles:
    # Calculate the z-score for the desired quantile
    z = norm.ppf(q)

    # Compute the quantile prediction per sample
    predictions = mu + z * sigma

    # Add predictions to the DataFrame
    predictions_NGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_NGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_NGB_df.head())

[iter 0] loss=9.9272 val_loss=0.0000 scale=2.0000 norm=3075.1018


[iter 100] loss=4.9970 val_loss=0.0000 scale=0.0000 norm=0.0002


        0.05        0.1       0.15        0.2        0.3        0.4  \
0   5.483289  11.626304  15.770968  19.065015  24.428848  29.012046   
1  19.676191  22.684392  24.714011  26.327088  28.953728  31.198095   
2  10.816611  13.203926  14.814636  16.094777  18.179285  19.960420   
3  10.205634  12.727897  14.429656  15.782159  17.984498  19.866314   
4   9.165893  11.917805  13.774508  15.250156  17.653015  19.706169   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  33.295851  37.579656  42.162854  47.526688  50.820735  54.965398   
1  33.295851  35.393607  37.637974  40.264614  41.877691  43.907310   
2  21.625203  23.289987  25.071122  27.155629  28.435770  30.046481   
3  21.625203  23.384092  25.265909  27.468247  28.820751  30.522510   
4  21.625203  23.544237  25.597392  28.000251  29.475899  31.332601   

        0.95  Actual  
0  61.108414    44.0  
1  46.915511    29.0  
2  32.433796    23.0  
3  33.044772    26.0  
4  34.084513    30.0  


In [39]:
quantile_regression_print(predictions_NGB_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

First few actual values: [44. 29. 23. 26. 30.]
Coverage of 90% prediction interval: 84.93%


In [40]:
evaluation_features_print(predictions_NGB_df,x_test,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

In [41]:
evaluate_uncertainity_matrix(predictions_NGB_df, quantiles,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/NGBoost.xlsx", model_name='NGBoost')

Quantile crossing before sorting: 0 of 365 rows (0.0%)


{'empirical_quantiles': [np.float64(0.0547945205479452),
  np.float64(0.15616438356164383),
  np.float64(0.2876712328767123),
  np.float64(0.40273972602739727),
  np.float64(0.5068493150684932),
  np.float64(0.5589041095890411),
  np.float64(0.6602739726027397),
  np.float64(0.7397260273972602),
  np.float64(0.7945205479452054),
  np.float64(0.8383561643835616),
  np.float64(0.8547945205479452),
  np.float64(0.8657534246575342),
  np.float64(0.9041095890410958)],
 'average_interval_lengths': [np.float64(142.09727263283082),
  np.float64(216.3740349434108),
  np.float64(277.7130672877382)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(5.748500566936573),
 'average_crps': np.float64(128.47523895030952),
 'nll_values': array([ 3.94715144,  3.16738455,  2.82351438,  3.05515232,  3.55495771,
         3.87389364,  3.1502314 ,  3.05730519,  2.91443066,  3.92854618,
         2.95787658,  3.10528678,  3.03929342,  3.02308799,  3.1019617 ,
         9.0544769 , 73.19477714, 16.

In [42]:
calibrate_and_plot_intervals(predictions_NGB_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

The empirical coverage before calibration is: 84.15%
The empirical coverage after calibration is: 90.71%


In [43]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_NGB_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

# **Quantile Regression with Gradient Boosting**

In [44]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')


# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GBR_df = pd.DataFrame()

# Custom quantile loss function
def quantile_loss(q, y_true, y_pred):
    e = y_true - y_pred
    return np.mean(np.maximum(q * e, (q - 1) * e))

# Train a model for each quantile and make predictions using GradientBoostingRegressor
for q in quantiles:

    params = best_params.copy()

    params.update({
        'loss': 'quantile',
        'alpha': q,
        'random_state': 42
    })
    # Initialize the model with quantile-specific settings
    model = GradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GBR_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  21.160183  26.371139  30.107132  31.406973  35.254097  36.891530   
1  22.299379  23.889749  29.338163  30.304205  35.224519  35.780435   
2  15.290879  15.287892  18.841897  21.760840  23.695899  24.273076   
3  11.975848  12.362583  16.416559  16.525786  20.507951  23.529335   
4  15.208255  14.888208  19.128486  21.030816  23.045890  25.112351   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  36.945401  45.114945  45.115167  45.403161  53.745912  56.109646   
1  36.650365  41.111064  45.275849  47.120972  51.046352  66.170816   
2  25.505318  27.419093  27.034093  29.309967  29.269552  27.976056   
3  25.774762  28.019173  27.113534  37.951276  27.267271  27.406120   
4  28.146439  27.035355  27.812466  28.835456  30.918242  34.543486   

        0.95  Actual  
0  84.796870    44.0  
1  88.134049    29.0  
2  29.149485    23.0  
3  28.086325    26.0  
4  50.758311    30.0  


In [45]:
quantile_regression_print(predictions_GBR_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

First few actual values: [44. 29. 23. 26. 30.]
Coverage of 90% prediction interval: 78.08%


In [46]:
evaluation_features_print(predictions_GBR_df,x_test,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

In [47]:
evaluate_uncertainity_matrix(predictions_GBR_df, quantiles,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx", model_name='GBM')

Quantile crossing before sorting: 328 of 365 rows (89.86%)


{'empirical_quantiles': [np.float64(0.1589041095890411),
  np.float64(0.1917808219178082),
  np.float64(0.25205479452054796),
  np.float64(0.3041095890410959),
  np.float64(0.3780821917808219),
  np.float64(0.4767123287671233),
  np.float64(0.5589041095890411),
  np.float64(0.6904109589041096),
  np.float64(0.7315068493150685),
  np.float64(0.8164383561643835),
  np.float64(0.8602739726027397),
  np.float64(0.9013698630136986),
  np.float64(0.9397260273972603)],
 'average_interval_lengths': [np.float64(290.30089569205364),
  np.float64(379.69283914230965),
  np.float64(512.3277091081322)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(5.6488312707910575),
 'average_crps': np.float64(73.8614558223963),
 'nll_values': array([  3.94782894,   3.9883566 ,   2.54150996,   2.98569782,
          3.34500689,   3.32022845,   2.59420797,   4.16235726,
          3.49558942,   5.28373084,   3.47640901,   2.81118526,
          2.42695949,   2.2236069 ,   2.24539807,   5.19883327,
 

In [48]:
calibrate_and_plot_intervals(predictions_GBR_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

The empirical coverage before calibration is: 77.60%
The empirical coverage after calibration is: 92.35%


In [49]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GBR_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

# **Quantile Regression with CatBoost**

In [50]:
# Get the best hyperparameters for CatBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_CatBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'loss_function', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'loss_function': 'Quantile:alpha={}'.format(q),
        'random_seed': 42
    })

    # Initialize the model with the updated parameters
    model = CatBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train, verbose=0)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_CatBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_CatBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_CatBoost_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  23.642036  26.428214  32.762303  35.147330  36.832346  44.187277   
1  23.265099  23.091131  30.330443  30.622764  33.596650  34.043377   
2  15.544249  17.790143  18.373304  20.838016  21.547335  25.096334   
3  12.890832  16.087624  15.700891  18.377578  19.272003  22.463535   
4  18.542788  20.918893  20.351004  20.794736  21.795077  26.826918   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  38.506689  46.580083  49.707195  44.464517  51.836242  81.160588   
1  35.490014  38.703844  41.970718  44.475307  44.638786  85.579677   
2  29.114084  27.031554  29.636236  33.060590  26.280423  28.326408   
3  24.041034  25.862987  23.867701  32.409043  37.412997  29.752440   
4  27.117126  28.229188  29.964664  34.071550  41.343730  86.123799   

        0.95  Actual  
0  88.009372    44.0  
1  76.875725    29.0  
2  71.673475    23.0  
3  31.988913    26.0  
4  81.823243    30.0  


In [51]:
quantile_regression_print(predictions_CatBoost_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

First few actual values: [44. 29. 23. 26. 30.]
Coverage of 90% prediction interval: 78.63%


In [52]:
evaluation_features_print(predictions_CatBoost_df,x_test,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

In [53]:
evaluate_uncertainity_matrix(predictions_CatBoost_df, quantiles,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/CatBoost.xlsx", model_name='CatBoost')

Quantile crossing before sorting: 339 of 365 rows (92.88%)


{'empirical_quantiles': [np.float64(0.1506849315068493),
  np.float64(0.21643835616438356),
  np.float64(0.27123287671232876),
  np.float64(0.32602739726027397),
  np.float64(0.37534246575342467),
  np.float64(0.44931506849315067),
  np.float64(0.5589041095890411),
  np.float64(0.6383561643835617),
  np.float64(0.7095890410958904),
  np.float64(0.810958904109589),
  np.float64(0.8547945205479452),
  np.float64(0.9095890410958904),
  np.float64(0.936986301369863)],
 'average_interval_lengths': [np.float64(198.76068531910897),
  np.float64(443.0361754773232),
  np.float64(559.6357305210513)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(6.2237459537338795),
 'average_crps': np.float64(71.4772277311023),
 'nll_values': array([  3.89279205,   3.92149124,   3.77427933,   2.96863055,
          3.95131361,   3.89117981,   3.9558347 ,   4.04816127,
          3.2310424 ,   5.44206689,   3.8924127 ,   3.26441704,
          3.58514718,   2.43307005,   2.41468415,   4.9475655 ,


In [54]:
calibrate_and_plot_intervals(predictions_CatBoost_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

The empirical coverage before calibration is: 79.23%
The empirical coverage after calibration is: 93.99%


In [55]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_CatBoost_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

# **Quantile Regression with HistGradientBoosting**

In [56]:

# Assume best_params is a dictionary containing the best parameters for HistGradientBoostingRegressor
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_HGBR_df = pd.DataFrame()

# Train a model for each quantile and make predictions using HistGradientBoostingRegressor
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'loss': 'quantile',
        'quantile': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = HistGradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_HGBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_HGBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_HGBR_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  24.200896  29.238826  32.911321  29.557641  33.450747  33.628359   
1  29.209590  31.719721  35.669595  33.941373  34.207831  33.244688   
2  21.222016  25.812897  22.495241  27.837464  27.447627  28.165309   
3  10.722938  13.019049  17.072279  19.968605  19.656350  21.771206   
4  19.578573  20.141021  21.530401  21.208974  23.226074  23.188754   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  39.463480  33.039455  39.535060  46.843556  63.523052  61.053614   
1  36.785412  34.731401  41.519561  47.079353  74.493305  59.027832   
2  28.754523  30.830225  32.948810  36.256587  47.847386  46.683736   
3  22.631514  24.847380  29.053462  29.664455  46.144148  41.165961   
4  23.599613  25.100103  25.930271  37.194076  47.847386  46.683736   

        0.95  Actual  
0  60.021967    44.0  
1  64.001091    29.0  
2  37.893044    23.0  
3  31.848324    26.0  
4  37.893044    30.0  


In [57]:
quantile_regression_print(predictions_HGBR_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/HGBM.xlsx")

First few actual values: [44. 29. 23. 26. 30.]
Coverage of 90% prediction interval: 76.99%


In [58]:
evaluation_features_print(predictions_HGBR_df,x_test,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/HGBM.xlsx")

In [59]:
evaluate_uncertainity_matrix(predictions_HGBR_df, quantiles,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/HGBM.xlsx", model_name='HGBM')

Quantile crossing before sorting: 363 of 365 rows (99.45%)


{'empirical_quantiles': [np.float64(0.1917808219178082),
  np.float64(0.2547945205479452),
  np.float64(0.31232876712328766),
  np.float64(0.3589041095890411),
  np.float64(0.41643835616438357),
  np.float64(0.5068493150684932),
  np.float64(0.5780821917808219),
  np.float64(0.6410958904109589),
  np.float64(0.7178082191780822),
  np.float64(0.8931506849315068),
  np.float64(0.9260273972602739),
  np.float64(0.9424657534246575),
  np.float64(0.9616438356164384)],
 'average_interval_lengths': [np.float64(216.3004572981731),
  np.float64(450.7463203841923),
  np.float64(523.114273023157)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(4.82892524601515),
 'average_crps': np.float64(79.67446779915612),
 'nll_values': array([ 3.77637594,  3.65846883,  3.2627665 ,  3.34438662,  3.34728339,
         2.95856293,  3.28547838,  3.98734234,  3.44058298,  3.83839716,
         3.36553023,  3.33980675,  3.37021922,  3.26364146,  3.26173858,
         6.93039509,  8.46944581, 13.5564

In [60]:
calibrate_and_plot_intervals(predictions_HGBR_df, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/HGBM.xlsx")

The empirical coverage before calibration is: 76.50%
The empirical coverage after calibration is: 93.99%


In [61]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_HGBR_df, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/HGBM.xlsx")

# **Quantile Regression with TabNet**

In [62]:
best_params = get_best_model_params(best_scores_autosampler, 'TabNet')

# Quantiles
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Storage
models_quantile = {}
predictions_TabNet_df = pd.DataFrame()

# Convert data to numpy (TabNet requirement)
X_train_np = np.array(X_train)
y_train_np = np.array(y_train).reshape(-1, 1)
X_test_np  = np.array(X_test)

# Quantile loss (pinball loss)
def quantile_loss(q):
    def loss(y_pred, y_true):
        error = y_true - y_pred
        return torch.mean(torch.maximum(q * error, (q - 1) * error))
    return loss

# FIX Q3: early stopping on a 20% validation split of TRAIN, scored with the pinball loss of
# the quantile being trained (was: eval_set = the training data itself, scored with MSE,
# so early stopping never triggered).
from pytorch_tabnet.metrics import Metric
X_fit_np, X_val_np, y_fit_np, y_val_np = train_test_split(X_train_np, y_train_np, test_size=0.2, random_state=42)

def pinball_metric(q):
    class Pinball(Metric):
        def __init__(self):
            self._name = f'pinball_{q}'
            self._maximize = False
        def __call__(self, y_true, y_score, weights=None):  # FIX B8: pytorch-tabnet2 also passes sample weights
            e = np.asarray(y_true, dtype=float) - np.asarray(y_score, dtype=float)
            return float(np.mean(np.maximum(q * e, (q - 1) * e)))
    return Pinball

# Train one TabNet per quantile
for q in quantiles:
    model = TabNetRegressor(
        **best_params,
    )

    model.fit(
        X_fit_np,
        y_fit_np,
        eval_set=[(X_val_np, y_val_np)],
        eval_metric=[pinball_metric(q)],
        loss_fn=quantile_loss(q),
        max_epochs=200,
        patience=30,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False
    )

    # Store model
    models_quantile[q] = model

    # Predict
    preds = model.predict(X_test_np).ravel()
    predictions_TabNet_df[q] = preds

# Add actual values
predictions_TabNet_df['Actual'] = y_test.values.ravel()

print(predictions_TabNet_df.head())


Early stopping occurred at epoch 98 with best_epoch = 68 and best_val_0_pinball_0.05 = 23.20421



Early stopping occurred at epoch 98 with best_epoch = 68 and best_val_0_pinball_0.1 = 42.73689



Early stopping occurred at epoch 98 with best_epoch = 68 and best_val_0_pinball_0.15 = 60.79803



Early stopping occurred at epoch 112 with best_epoch = 82 and best_val_0_pinball_0.2 = 71.60213



Early stopping occurred at epoch 145 with best_epoch = 115 and best_val_0_pinball_0.3 = 95.44384



Early stopping occurred at epoch 157 with best_epoch = 127 and best_val_0_pinball_0.4 = 119.17847


... (3 outputs left out)


Early stopping occurred at epoch 197 with best_epoch = 167 and best_val_0_pinball_0.8 = 130.55083



Early stopping occurred at epoch 146 with best_epoch = 116 and best_val_0_pinball_0.85 = 138.46753



Early stopping occurred at epoch 194 with best_epoch = 164 and best_val_0_pinball_0.9 = 124.2601



Early stopping occurred at epoch 147 with best_epoch = 117 and best_val_0_pinball_0.95 = 128.29308
       0.05       0.1       0.15        0.2        0.3        0.4        0.5  \
0  6.744476  5.674891  10.356874  17.267376  17.464125  39.024849  30.377903   
1  6.616960  5.545705  10.058609  16.987774  16.738224  38.542099  29.403273   
2  6.336331  4.927850   9.222218  14.465822  12.105244  28.392296  24.633003   
3  6.316862  5.018579   9.247002  15.278797  11.812931  26.672626  23.855572   
4  6.458909  5.194532   9.582883  15.945869  12.460605  28.520874  25.138191   

         0.6        0.7        0.8       0.85         0.9        0.95  Actual  
0  37.372051  37.525394  33.741844  62.500927  110.696976  243.015396    44.0  
1  36.540894  36.536934  32.981487  61.806137  109.432724  241.554108    29.0  
2  30.656599  30.779266  26.063389  50.105637   89.175461  226.600250    23.0  
3  33.812366  32.138111  29.594408  57.307213  102.781097  231.009369    26.0  
4  35.582146  33.64

In [63]:
quantile_regression_print(predictions_TabNet_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/TabNet.xlsx")

First few actual values: [44. 29. 23. 26. 30.]
Coverage of 90% prediction interval: 89.32%


In [64]:
evaluation_features_print(predictions_TabNet_df,x_test,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/TabNet.xlsx")

In [65]:
evaluate_uncertainity_matrix(predictions_TabNet_df, quantiles, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/TabNet.xlsx", model_name='TabNet')

Quantile crossing before sorting: 360 of 365 rows (98.63%)


{'empirical_quantiles': [np.float64(0.09315068493150686),
  np.float64(0.12054794520547946),
  np.float64(0.2219178082191781),
  np.float64(0.29863013698630136),
  np.float64(0.4465753424657534),
  np.float64(0.6657534246575343),
  np.float64(0.7424657534246575),
  np.float64(0.8027397260273973),
  np.float64(0.8383561643835616),
  np.float64(0.8904109589041096),
  np.float64(0.936986301369863),
  np.float64(0.9671232876712329),
  np.float64(0.9863013698630136)],
 'average_interval_lengths': [np.float64(243.33325552221848),
  np.float64(412.6144330834689),
  np.float64(597.1554830185355)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(5.575143207722362),
 'average_crps': np.float64(69.9656315841675),
 'nll_values': array([ 5.20774412,  5.19354736,  5.13037396,  5.14868214,  5.16501652,
         5.16664207,  5.13948086,  5.11258063,  5.08212615,  5.07917105,
         5.08618793,  5.07586131,  5.0911309 ,  5.07440348,  5.07830068,
         5.36094785,  9.66365237, 12.59

In [66]:
calibrate_and_plot_intervals(predictions_TabNet_df, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/TabNet.xlsx")

The empirical coverage before calibration is: 88.52%
The empirical coverage after calibration is: 91.80%


In [67]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_TabNet_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/TabNet.xlsx")

# **Quantile Regression with PGBM**

In [68]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [69]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual target values to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())

Training on CPU


Estimator 0/500, Train metric: 4832.2310
Estimator 1/500, Train metric: 4807.3794
Estimator 2/500, Train metric: 4782.7339
Estimator 3/500, Train metric: 4758.2944
Estimator 4/500, Train metric: 4734.0581
Estimator 5/500, Train metric: 4710.0234
Estimator 6/500, Train metric: 4686.1880
Estimator 7/500, Train metric: 4662.5513


Estimator 8/500, Train metric: 4639.1113
Estimator 9/500, Train metric: 4615.8677
Estimator 10/500, Train metric: 4592.8184
Estimator 11/500, Train metric: 4569.9619
Estimator 12/500, Train metric: 4547.2974
Estimator 13/500, Train metric: 4524.8228
Estimator 14/500, Train metric: 4502.5371
Estimator 15/500, Train metric: 4480.4395
Estimator 16/500, Train metric: 4458.5283
Estimator 17/500, Train metric: 4436.8018
Estimator 18/500, Train metric: 4415.2593


Estimator 19/500, Train metric: 4393.8916
Estimator 20/500, Train metric: 4372.7129
Estimator 21/500, Train metric: 4351.7139
Estimator 22/500, Train metric: 4330.8936
Estimator 23/500, Train metric: 4310.2437
Estimator 24/500, Train metric: 4289.7783
Estimator 25/500, Train metric: 4269.4790
Estimator 26/500, Train metric: 4249.3628
Estimator 27/500, Train metric: 4229.4170
Estimator 28/500, Train metric: 4209.6416
Estimator 29/500, Train metric: 4190.0381


Estimator 30/500, Train metric: 4170.6011
Estimator 31/500, Train metric: 4151.3252
Estimator 32/500, Train metric: 4132.2241
Estimator 33/500, Train metric: 4113.2876
Estimator 34/500, Train metric: 4094.5076
Estimator 35/500, Train metric: 4075.8958
Estimator 36/500, Train metric: 4057.4482
Estimator 37/500, Train metric: 4039.1624
Estimator 38/500, Train metric: 4021.0266
Estimator 39/500, Train metric: 4003.0554
Estimator 40/500, Train metric: 3985.2427


Estimator 41/500, Train metric: 3967.5879
Estimator 42/500, Train metric: 3950.0837
Estimator 43/500, Train metric: 3932.7368
Estimator 44/500, Train metric: 3915.5391
Estimator 45/500, Train metric: 3898.4878
Estimator 46/500, Train metric: 3881.5945
Estimator 47/500, Train metric: 3864.8511
Estimator 48/500, Train metric: 3848.2468
Estimator 49/500, Train metric: 3831.7966
Estimator 50/500, Train metric: 3815.4946
Estimator 51/500, Train metric: 3799.3376


... (53 outputs left out)

Estimator 480/500, Train metric: 2090.6692
Estimator 481/500, Train metric: 2090.3782
Estimator 482/500, Train metric: 2090.0693
Estimator 483/500, Train metric: 2089.6882
Estimator 484/500, Train metric: 2088.1738
Estimator 485/500, Train metric: 2087.8928
Estimator 486/500, Train metric: 2087.6155
Estimator 487/500, Train metric: 2086.1196


Estimator 488/500, Train metric: 2085.8442
Estimator 489/500, Train metric: 2085.5781
Estimator 490/500, Train metric: 2084.0981
Estimator 491/500, Train metric: 2083.8257
Estimator 492/500, Train metric: 2083.5635
Estimator 493/500, Train metric: 2082.0869
Estimator 494/500, Train metric: 2081.8113


Estimator 495/500, Train metric: 2081.5459
Estimator 496/500, Train metric: 2081.2756
Estimator 497/500, Train metric: 2079.8174
Estimator 498/500, Train metric: 2078.3765
Estimator 499/500, Train metric: 2076.9460


        0.05        0.1       0.15        0.2        0.3        0.4  \
0  42.317627  42.580471  42.799820  42.986034  43.636372  43.891960   
1  41.374252  42.301384  42.620876  42.867317  43.267574  43.522133   
2  28.471724  29.515415  29.981430  30.655924  31.062454  31.336077   
3  28.088083  28.901344  29.313597  29.743921  30.380558  30.694120   
4  29.597191  29.784525  29.985497  30.079283  30.427866  30.858833   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  44.063148  44.260803  44.499088  44.894718  45.346554  46.026077   
1  43.824627  44.029472  44.406349  45.003632  45.138027  45.823959   
2  31.441914  31.622396  31.810875  32.111771  32.356735  32.593292   
3  30.913490  31.289352  31.709581  32.268368  32.502934  32.937233   
4  31.167099  31.438446  31.607830  32.094898  32.243137  32.510990   

        0.95  Actual  
0  46.792320    44.0  
1  46.734184    29.0  
2  33.149940    23.0  
3  33.201416    26.0  
4  33.173508    30.0  


In [70]:
quantile_regression_print(predictions_PGBM_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/PGBM.xlsx")

First few actual values: [44. 29. 23. 26. 30.]
Coverage of 90% prediction interval: 5.75%


In [71]:
evaluation_features_print(predictions_PGBM_df,x_test,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/PGBM.xlsx")

In [72]:
evaluate_uncertainity_matrix(predictions_PGBM_df, quantiles, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/PGBM.xlsx", model_name='PGBM')

Quantile crossing before sorting: 0 of 365 rows (0.0%)


{'empirical_quantiles': [np.float64(0.7808219178082192),
  np.float64(0.7917808219178082),
  np.float64(0.7972602739726027),
  np.float64(0.810958904109589),
  np.float64(0.8136986301369863),
  np.float64(0.8164383561643835),
  np.float64(0.8246575342465754),
  np.float64(0.8246575342465754),
  np.float64(0.8273972602739726),
  np.float64(0.8356164383561644),
  np.float64(0.8383561643835616),
  np.float64(0.8383561643835616),
  np.float64(0.8383561643835616)],
 'average_interval_lengths': [np.float64(3.473537037470569),
  np.float64(5.952461671176022),
  np.float64(8.320836571471332)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(1762.9035953806492),
 'average_crps': np.float64(113.43190676248892),
 'nll_values': array([1.22765543e+00, 4.28007203e+01, 1.88910448e+01, 6.35638449e+00,
        1.57874595e+00, 1.27717561e+00, 8.08379699e+00, 5.94079616e+01,
        2.62243526e+00, 1.44897999e+00, 3.51813126e+01, 4.93917267e+01,
        1.42143062e+01, 1.40374108e+01, 2.8

In [73]:
calibrate_and_plot_intervals(predictions_PGBM_df, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/PGBM.xlsx")

The empirical coverage before calibration is: 7.65%
The empirical coverage after calibration is: 91.26%


In [74]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_PGBM_df,"./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/quantile_regression/PGBM.xlsx")